# 🛩️ F-18 DDPG v3 — Aggressive Heading-Change Training

**Key fixes in v3:**
- ✅ Redesigned action space: `(psi_dot_cmd, bank_rate_cmd, hdot_trim)` — no more altitude blowup
- ✅ Proportional altitude hold autopilot keeps aircraft within ±50m
- ✅ Normalised observations fed to networks
- ✅ Curriculum: Phase 0 (fly level) → Phase 1 (full objectives)
- ✅ nz sweet-spot reward 4.5–6.8g, hard cliff above 7g
- ✅ Decoupled training and plotting — run them independently

**Notebook structure:**
1. Install & Imports
2. Physics & Config
3. Dynamics Model (F-18 6-DOF)
4. Reward Function
5. Environment
6. DDPG Networks
7. **TRAINING** ← run this cell to train, saves `.pt` checkpoints
8. **PLOTTING** ← run this cell independently after training (or load checkpoint)
9. Constraint Report

## 1. Install & Imports

In [3]:
# ── Install (only needed once per Colab session) ──────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch', '--quiet'], check=True)
print('Done')

Done


In [4]:
import math, random, copy, time, os, warnings, json
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import Image, display

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__}  |  device = {DEVICE}')

PyTorch 2.10.0+cu128  |  device = cuda


## 2. Physics Constants & Config

In [5]:
# ══ PHYSICS ══════════════════════════════════════════════════════════════════
G            = 9.81
V_CRUISE     = 260.0      # m/s
Z_CRUISE     = 5000.0     # m
Z_BAND       = 50.0       # m   HARD altitude wall
NZ_MIN       = 3.0        # g   floor
NZ_MAX       = 7.0        # g   structural ceiling
NZ_SWEET_LO  = 4.5        # g   reward band low
NZ_SWEET_HI  = 6.8        # g   reward band high
PHI_LIMIT    = 80.0       # deg structural bank limit
PHI_AGGR     = 65.0       # deg aggression threshold
DT           = 0.5        # s   simulation timestep

# Derived
PHI_OPT_DEG  = min(PHI_LIMIT, math.degrees(math.acos(1.0/NZ_MAX)))
PSI_DOT_MAX  = math.degrees(G * math.tan(math.radians(PHI_OPT_DEG)) / V_CRUISE)
R_OPT        = V_CRUISE / math.radians(PSI_DOT_MAX)

STATE_DIM    = 12
ACTION_DIM   = 3

# Action space bounds (FIX-1: direct rate commands)
A_PSI_DOT    = PSI_DOT_MAX   # deg/s max turn rate
A_BANK_RATE  = 60.0          # deg/s max bank rate
A_DZ_TRIM    = 3.0           # m/s   altitude trim authority (tight!)

# Altitude autopilot gains (FIX-2)
KH_HOLD      = 0.8
KH_MAX_HDOT  = 5.0

HEADINGS = [0, 30, 60, 90, 120, 150, 180]

print(f'PHI_OPT  = {PHI_OPT_DEG:.1f} deg')
print(f'PSI_DOT  = {PSI_DOT_MAX:.1f} deg/s')
print(f'R_OPT    = {R_OPT:.0f} m')
print(f'Actions  : psi_dot ±{A_PSI_DOT:.1f}°/s | bank_rate ±{A_BANK_RATE:.0f}°/s | hdot_trim ±{A_DZ_TRIM:.1f} m/s')

PHI_OPT  = 80.0 deg
PSI_DOT  = 12.3 deg/s
R_OPT    = 1215 m
Actions  : psi_dot ±12.3°/s | bank_rate ±60°/s | hdot_trim ±3.0 m/s


## 3. Maneuver Spec

In [6]:
@dataclass
class ManeuverSpec:
    psi0:float; psi1:float; delta:float
    key:str; label:str; category:str
    phi_opt:float; psi_dot_opt:float; R:float
    t_opt:float; t_limit:float; nz_design:float
    x_target:float; y_target:float; z_target:float=Z_CRUISE
    success_herr:float=5.0
    success_hold:int=4
    alt_strikes:int=3
    extra:dict=field(default_factory=dict)


def _arc_endpoint(psi0_deg, delta_deg, R):
    psi0_r=math.radians(psi0_deg); d_r=math.radians(delta_deg)
    x_arc=R*(1-math.cos(d_r)); y_arc=R*math.sin(d_r)
    s0=math.sin(psi0_r); c0=math.cos(psi0_r)
    return x_arc*c0+y_arc*s0, -x_arc*s0+y_arc*c0


def build_specs():
    specs={}
    for i,psi0 in enumerate(HEADINGS):
        for psi1 in HEADINGS[i+1:]:
            delta=float(psi1-psi0)
            phi_r=math.radians(PHI_OPT_DEG)
            pd=math.degrees(G*math.tan(phi_r)/V_CRUISE)
            R=V_CRUISE/math.radians(pd)
            t_opt=delta/pd
            xt,yt=_arc_endpoint(psi0,delta,R)
            dist=math.hypot(xt,yt)
            cat='nav' if delta<=60 else ('aggressive' if delta<=120 else 'extreme')
            key=f'H{psi0:03d}_{psi1:03d}'
            specs[key]=ManeuverSpec(
                psi0=float(psi0), psi1=float(psi1), delta=delta,
                key=key, label=f'{psi0}°→{psi1}° ({delta:.0f}°)',
                category=cat, phi_opt=PHI_OPT_DEG, psi_dot_opt=pd,
                R=R, t_opt=t_opt, t_limit=max(2.5*t_opt,15.0),
                nz_design=1/math.cos(phi_r),
                x_target=xt, y_target=yt,
                success_herr=max(3.0,delta*0.07),
                extra=dict(dist_init=dist)
            )
    print(f'Built {len(specs)} specs')
    return specs

ALL_SPECS = build_specs()

# Quick peek at H000_030
sp1=ALL_SPECS['H000_030']
sp2=ALL_SPECS['H000_060']
sp3=ALL_SPECS['H000_090']
sp4=ALL_SPECS['H000_180']
print(f'\nH000_030: T_opt={sp1.t_opt:.2f}s | arc=({sp1.x_target:.0f},{sp1.y_target:.0f})m | dist={sp1.extra["dist_init"]:.0f}m')
print(f'\nH000_060: T_opt={sp2.t_opt:.2f}s | arc=({sp2.x_target:.0f},{sp2.y_target:.0f})m | dist={sp2.extra["dist_init"]:.0f}m')
print(f'\nH000_090: T_opt={sp3.t_opt:.2f}s | arc=({sp3.x_target:.0f},{sp3.y_target:.0f})m | dist={sp3.extra["dist_init"]:.0f}m')
print(f'\nH000_180: T_opt={sp4.t_opt:.2f}s | arc=({sp4.x_target:.0f},{sp4.y_target:.0f})m | dist={sp4.extra["dist_init"]:.0f}m')

Built 21 specs

H000_030: T_opt=2.45s | arc=(163,608)m | dist=629m

H000_060: T_opt=4.89s | arc=(608,1052)m | dist=1215m

H000_090: T_opt=7.34s | arc=(1215,1215)m | dist=1718m

H000_180: T_opt=14.68s | arc=(2430,0)m | dist=2430m


## 4. F-18 6-DOF Dynamics

In [7]:
class F18_6DOF:
    MASS=16651; Ixx=23000; Iyy=169000; Izz=185000; Ixz=1500
    WING_AREA=37.16; SPAN=11.43; CHORD=3.51
    RHO_SL=1.225; H_SCALE=8500
    CL0=0.10; CL_a=5.0; CL_q=4.0
    CD0=0.022; K_ind=0.16
    CY_b=-0.90; CY_r=0.30
    Cl_p=-0.45; Cl_r=0.18; Cl_b=-0.08
    Cm_a=-1.20; Cm_q=-8.0; Cm0=0.02
    Cn_b=0.18; Cn_p=-0.08; Cn_r=-0.25
    V_MIN=150.0; V_MAX=450.0; NZ_CAP=9.0
    P_MAX=100.0; Q_MAX=25.0; R_MAX=20.0
    THETA_MAX=25.0; THETA_MIN=-15.0

    def __init__(self, dt=DT): self.dt=dt; self.reset()

    def reset(self, psi0=0.0):
        self.x=0.; self.y=0.; self.z=Z_CRUISE
        self.psi=float(psi0); self.theta=0.; self.phi=0.
        self.u=V_CRUISE; self.v=0.; self.w=0.
        self.p=0.; self.q=0.; self.r=0.
        self.V=V_CRUISE; self.alpha=0.; self.beta=0.; self.nz=1.
        self.steps=0; self.elapsed=0.

    def _rho(self): return self.RHO_SL*math.exp(-max(self.z,0)/self.H_SCALE)

    def _aero(self):
        rho=self._rho(); V=max(self.V,1.)
        self.alpha=math.degrees(math.atan2(self.w,self.u))
        self.beta =math.degrees(math.asin(np.clip(self.v/V,-1.,1.)))
        ar=math.radians(self.alpha); br=math.radians(self.beta)
        qb=0.5*rho*V**2; S=self.WING_AREA; b=self.SPAN; c=self.CHORD
        pn=math.radians(self.p)*b/(2*V)
        qn=math.radians(self.q)*c/(2*V)
        rn=math.radians(self.r)*b/(2*V)
        CL=min(self.CL0+self.CL_a*ar+self.CL_q*qn, 1.8)
        CD=self.CD0+self.K_ind*CL**2
        CY=self.CY_b*br+self.CY_r*rn
        Cl_=self.Cl_p*pn+self.Cl_r*rn+self.Cl_b*br
        Cm_=self.Cm0+self.Cm_a*ar+self.Cm_q*qn
        Cn_=self.Cn_b*br+self.Cn_p*pn+self.Cn_r*rn
        ca=math.cos(ar); sa=math.sin(ar); cb=math.cos(br)
        L=qb*S*CL; D=qb*S*CD; Y=qb*S*CY
        Xa=-D*ca*cb+L*sa; Ya=Y; Za=-D*sa*cb-L*ca
        Lm=qb*S*b*Cl_; Ma=qb*S*c*Cm_; Na=qb*S*b*Cn_
        return Xa,Ya,Za,Lm,Ma,Na,L

    def step(self, psi_dot_cmd_deg, bank_rate_cmd_deg, hdot_cmd_ms):
        """
        FIX-1 + FIX-2: Direct rate commands + proportional altitude hold.
        psi_dot_cmd_deg  : desired turn rate [deg/s]
        bank_rate_cmd_deg: bank rate nudge   [deg/s]
        hdot_cmd_ms      : altitude trim     [m/s]   max ±3 m/s
        """
        self.steps+=1; self.elapsed+=self.dt; dt=self.dt

        # Bank autopilot
        phi_des_from_turn = math.degrees(
            math.atan(math.radians(psi_dot_cmd_deg)*self.V/G))
        phi_des = np.clip(phi_des_from_turn + bank_rate_cmd_deg*dt,
                          -PHI_LIMIT, PHI_LIMIT)
        dphi    = np.clip(phi_des-self.phi, -60.*dt, 60.*dt)
        self.phi= np.clip(self.phi+dphi, -PHI_LIMIT, PHI_LIMIT)

        # Altitude hold (FIX-2) — proportional + agent trim
        z_err_now  = Z_CRUISE - self.z
        hdot_auto  = KH_HOLD * z_err_now
        hdot_total = np.clip(hdot_auto + hdot_cmd_ms, -KH_MAX_HDOT, KH_MAX_HDOT)
        theta_des  = np.clip(math.degrees(math.asin(
            np.clip(hdot_total/max(self.V,1.), -0.3, 0.3))),
            self.THETA_MIN, self.THETA_MAX)
        dtheta     = np.clip(theta_des-self.theta, -15.*dt, 15.*dt)
        self.theta = np.clip(self.theta+dtheta, self.THETA_MIN, self.THETA_MAX)

        Xa,Ya,Za,Lm,Ma,Na,Lift = self._aero()
        phi_r=math.radians(self.phi); theta_r=math.radians(self.theta)
        Gx=-G*math.sin(theta_r)
        Gy= G*math.cos(theta_r)*math.sin(phi_r)
        Gz= G*math.cos(theta_r)*math.cos(phi_r)
        T=np.clip(self.MASS*(G*math.sin(theta_r)+0.3*(V_CRUISE-self.V)),0.,self.MASS*50.)
        pr=math.radians(self.p); qr=math.radians(self.q); rr=math.radians(self.r)
        ax=(Xa+T)/self.MASS+Gx-(qr*self.w-rr*self.v)
        ay=Ya/self.MASS+Gy-(rr*self.u-pr*self.w)
        az=Za/self.MASS+Gz-(pr*self.v-qr*self.u)
        self.u=np.clip(self.u+ax*dt,-self.V_MAX,self.V_MAX)
        self.v=np.clip(self.v+ay*dt,-50.,50.)
        self.w=np.clip(self.w+az*dt,-60.,60.)

        Ix=self.Ixx; Iy=self.Iyy; Iz=self.Izz; Ixz_=self.Ixz
        Gam=Ix*Iz-Ixz_**2
        pd=(Iz*Lm+Ixz_*Na-(Iz*(Iz-Iy)*rr+Ixz_*(Ix-Iy+Iz)*pr)*qr)/Gam
        qd=(Ma-(Ix-Iz)*pr*rr-Ixz_*(pr**2-rr**2))/Iy
        rd=(Ix*Na+Ixz_*Lm+(Ix*(Ix-Iy)*pr+Ixz_*(Ix-Iy+Iz)*rr)*qr)/Gam
        self.p=np.clip(self.p+math.degrees(pd)*dt,-self.P_MAX,self.P_MAX)
        self.q=np.clip(self.q+math.degrees(qd)*dt,-self.Q_MAX,self.Q_MAX)
        self.r=np.clip(self.r+math.degrees(rd)*dt,-self.R_MAX,self.R_MAX)

        phi_r=math.radians(self.phi); theta_r=math.radians(self.theta)
        pr=math.radians(self.p); qr=math.radians(self.q); rr=math.radians(self.r)
        ct=math.cos(theta_r); cf=math.cos(phi_r); sf=math.sin(phi_r)
        psi_dot2=(qr*sf+rr*cf)/(ct+1e-9)
        theta_d2=qr*cf-rr*sf
        phi_d2  =pr+(qr*sf+rr*cf)*math.tan(theta_r)
        self.psi  =(self.psi+math.degrees(psi_dot2)*dt)%360.
        self.theta=np.clip(self.theta+math.degrees(theta_d2)*dt,self.THETA_MIN,self.THETA_MAX)
        self.phi  =np.clip(self.phi+math.degrees(phi_d2)*dt,-PHI_LIMIT,PHI_LIMIT)

        phi_r=math.radians(self.phi); theta_r=math.radians(self.theta)
        psi_r=math.radians(self.psi)
        cp=math.cos(psi_r); sp=math.sin(psi_r)
        ct2=math.cos(theta_r); st2=math.sin(theta_r)
        cf2=math.cos(phi_r); sf2=math.sin(phi_r)
        vn=(ct2*cp)*self.u+(sf2*st2*cp-cf2*sp)*self.v+(cf2*st2*cp+sf2*sp)*self.w
        ve=(ct2*sp)*self.u+(sf2*st2*sp+cf2*cp)*self.v+(cf2*st2*sp-sf2*cp)*self.w
        vd=(-st2)*self.u+(sf2*ct2)*self.v+(cf2*ct2)*self.w
        self.x+=ve*dt; self.y+=vn*dt; self.z-=vd*dt

        self.nz=np.clip(Lift*math.cos(phi_r)/(self.MASS*G), 0.1, self.NZ_CAP)
        self.V =math.sqrt(self.u**2+self.v**2+self.w**2)
        return self.sv()

    def sv(self):
        return dict(x=self.x,y=self.y,z=self.z,
                    psi=self.psi,theta=self.theta,phi=self.phi,
                    u=self.u,v=self.v,w=self.w,p=self.p,q=self.q,r=self.r,
                    V=self.V,alpha=self.alpha,beta=self.beta,nz=self.nz,
                    elapsed=self.elapsed)

    def state_vec(self):
        # FIX-6: normalised to ~[-1,+1]
        return np.array([
            self.x/5000., self.y/5000., (self.z-Z_CRUISE)/Z_BAND,
            self.psi/180., self.theta/30., self.phi/PHI_LIMIT,
            (self.u-V_CRUISE)/100., self.v/50., self.w/30.,
            self.p/self.P_MAX, self.q/self.Q_MAX, self.r/self.R_MAX
        ], dtype=np.float32)

print('F18_6DOF defined ✓')

F18_6DOF defined ✓


## 5. Reward Function

In [8]:
def herr(psi, target):
    e=target-psi
    while e> 180.: e-=360.
    while e<=-180.: e+=360.
    return e


def compute_reward(sv_prev, sv, spec, action, done, cause, curriculum_phase):
    """
    Reward components — all normalised to [-1,+1] before weighting.

    PHASE 0 (curriculum_phase=0): altitude + heading only → learn to fly level
    PHASE 1 (curriculum_phase=1): full reward

    Weights:
      w_alt       = 50   HARD BARRIER (tanh-cliff at ±50m)
      w_heading   = 30   Gaussian heading error
      w_pos       = 80   exp-kernel to arc endpoint
      w_approach  = 15   velocity toward target
      w_nz        = 20   load factor band 3–7g
      w_bank      = 20   steep bank φ>65°
      w_speed     = 10   speed band [230,290]
      w_time      = 15   time urgency
    """
    he_abs   = abs(herr(sv['psi'], spec.psi1))
    z_err_abs= abs(sv['z'] - Z_CRUISE)
    t        = sv['elapsed']
    dist_init= spec.extra.get('dist_init', 1000.)
    dist_now = math.hypot(sv['x']-spec.x_target, sv['y']-spec.y_target)

    # 1. Altitude HARD WALL
    z_norm = z_err_abs / Z_BAND
    if z_err_abs <= Z_BAND:
        R_alt = 1.0 - z_norm**2  # Quadratic shaping
    else:
        excess = z_err_abs - Z_BAND
        R_alt  = -1.0 - (excess / 10.0)**2 # Severe penalty for leaving band
    w_alt = 100.0

    # 2. Heading Gaussian
    sigma = max(5., spec.delta/8.)
    R_heading = math.exp(-0.5*(he_abs/sigma)**2)
    w_heading = 40.

    # 3. Position exp kernel
    dist_prev = math.hypot(sv_prev['x'] - spec.x_target, sv_prev['y'] - spec.y_target)
    dist_delta = dist_prev - dist_now

    R_pos = float(np.clip(dist_delta / 130.0, -1.0, 1.0))
    w_pos = 150.  # 🔥 Massive weight to drag it to the coordinate

    # 4. Approach velocity toward arc target
    psi_r   = math.radians(sv['psi'])
    vx      = sv['V']*math.sin(psi_r)
    vy      = sv['V']*math.cos(psi_r)
    dx_t    = spec.x_target - sv['x']
    dy_t    = spec.y_target - sv['y']
    d_horiz = math.hypot(dx_t, dy_t) + 1e-6
    v_toward= (vx*dx_t + vy*dy_t) / d_horiz
    R_approach = float(np.clip(v_toward/V_CRUISE, -1., 1.))
    w_approach = 20.0 if curriculum_phase >= 1 else 0.0
    # 5. Load factor band [3–7g]
    nz = sv['nz']
    if nz < NZ_SWEET_LO:
        # Heavily penalize not pulling enough Gs to make the turn
        R_nz = -2.0 * (NZ_SWEET_LO - nz)
    elif nz <= NZ_SWEET_HI:
        R_nz = 1.0 # Sweet spot
    elif nz <= NZ_MAX:
        R_nz = 1.0 - (nz - NZ_SWEET_HI) / (NZ_MAX - NZ_SWEET_HI)
    else:
        R_nz = -5.0 * (nz - NZ_MAX) # Structural over-stress
    w_nz = 60.0 # Increased to drive aggressive maneuvering

    # 6. Contextual Bank Reward
    # Only reward banking if there is a significant heading error remaining
    phi_abs = abs(sv['phi'])
    if he_abs > 10.0:  # Only care about bank if we still need to turn
        if phi_abs >= PHI_AGGR:
            R_bank = (phi_abs - PHI_AGGR) / (PHI_LIMIT - PHI_AGGR)
        else:
            R_bank = -0.2 * (1. - phi_abs / PHI_AGGR)
    else:
        R_bank = -phi_abs / PHI_LIMIT # Penalize banking when heading is nearly reached
    w_bank = 20.0

    # 7. Speed band [230,290] m/s
    v_dev  = max(0., abs(sv['V']-V_CRUISE)-30.)
    R_speed= float(np.clip(1.-(v_dev/30.)**2, -1., 1.))
    w_speed= 10.

    # 8. Time urgency
    if curriculum_phase >= 1:
        R_time = 1.-t/spec.t_opt if t<=spec.t_opt else -((t-spec.t_opt)/spec.t_opt)**2
    else:
        R_time = 0.
    w_time = 20.

    # Terminal
    R_terminal = 0.
    if cause == 'success':
        # Massive base bonus for reaching the 200m radius
        base_bonus = 3000.
        # Extra points for hitting the exact dead-center of the coordinate
        center_bonus = 500. * max(0., 1. - dist_now / 200.0)
        # Extra points if the final heading also aligns nicely
        heading_bonus = 500. * max(0., 1. - he_abs / 10.0)
        # Extra points for flying fast
        time_bonus = max(0., 500. * (1. - t / spec.t_opt)) if t <= spec.t_opt else 0.

        R_terminal = base_bonus + center_bonus + heading_bonus + time_bonus

    elif cause == 'alt_violation':
        R_terminal = -1500. # Keep this harsh so it stops diving
    elif cause == 'timeout':
        # Partial credit if it ran out of time but got close to the neighborhood
        R_terminal = -200. + 300. * max(0., 1. - dist_now / dist_init)
    elif cause == 'crash':
        R_terminal = -2000.

    total = (w_alt*R_alt + w_heading*R_heading + w_pos*R_pos
             + w_approach*R_approach + w_nz*R_nz + w_bank*R_bank
             + w_speed*R_speed + w_time*R_time + R_terminal)

    info=dict(alt=R_alt*w_alt, heading=R_heading*w_heading,
              pos=R_pos*w_pos, approach=R_approach*w_approach,
              nz=R_nz*w_nz, bank=R_bank*w_bank,
              speed=R_speed*w_speed, time=R_time*w_time,
              terminal=R_terminal)
    return float(total), info

print('Reward function defined ✓')

Reward function defined ✓


## 6. Environment

In [9]:
class HeadingChangeEnv:
    def __init__(self, spec, curriculum_phase=1):
        self.spec=spec; self.dyn=F18_6DOF()
        self.curriculum_phase=curriculum_phase
        self._sv=self._sv_prev=None
        self._suc_ctr=self._alt_ctr=0

    def reset(self):
        self.dyn.reset(psi0=self.spec.psi0)
        self._sv=self.dyn.sv(); self._sv_prev=self._sv.copy()
        self._suc_ctr=self._alt_ctr=0
        return self.dyn.state_vec()

    def step(self, action):
        a=np.clip(action,-1.,1.)
        # FIX-1: map [-1,+1] → physical rates
        psi_dot_cmd   = float(a[0])*A_PSI_DOT
        bank_rate_cmd = float(a[1])*A_BANK_RATE
        hdot_cmd      = float(a[2])*A_DZ_TRIM

        self._sv_prev=self._sv.copy()
        self._sv=self.dyn.step(psi_dot_cmd, bank_rate_cmd, hdot_cmd)

        he_abs = abs(herr(self._sv['psi'], self.spec.psi1))
        z_err  = abs(self._sv['z'] - Z_CRUISE)
        t      = self._sv['elapsed']

        # Calculate distance to the target coordinate
        dist_now = math.hypot(self._sv['x'] - self.spec.x_target, self._sv['y'] - self.spec.y_target)

        self._alt_ctr = self._alt_ctr + 1 if z_err > Z_BAND else max(0, self._alt_ctr - 1)

        done = False
        cause = 'running'

        if self._sv['z'] < 3000. or self._sv['V'] < F18_6DOF.V_MIN:
            done, cause = True, 'crash'
        elif self._alt_ctr >= self.spec.alt_strikes:
            done, cause = True, 'alt_violation'
        elif t >= self.spec.t_limit:
            done, cause = True, 'timeout'
        else:
            # 🎯 NEW SUCCESS DEFINITION: Reach the neighborhood (200m radius)
            target_radius = 200.0

            if dist_now <= target_radius and z_err <= Z_BAND:
                self._suc_ctr += 1
                if self._suc_ctr >= self.spec.success_hold:
                    done, cause = True, 'success'
            else:
                self._suc_ctr = 0

        reward,rew_info=compute_reward(
            self._sv_prev,self._sv,self.spec,a,done,cause,self.curriculum_phase)

        obs=self.dyn.state_vec()
        info=dict(cause=cause,herr=he_abs,z_err=z_err,
                  z=self._sv['z'],V=self._sv['V'],
                  phi=self._sv['phi'],nz=self._sv['nz'],
                  t=t,x=self._sv['x'],y=self._sv['y'],
                  psi=self._sv['psi'],rew_info=rew_info)
        return obs,reward,done,info

    @property
    def state(self): return self._sv.copy()

print('Environment defined ✓')

Environment defined ✓


## 7. DDPG Networks & Agent

In [10]:
def mlp(in_d, out_d, hidden=(512,512)):
    layers,prev=[],in_d
    for h in hidden:
        layers+=[nn.Linear(prev,h), nn.LayerNorm(h), nn.ReLU()]
        prev=h
    layers.append(nn.Linear(prev,out_d))
    return nn.Sequential(*layers)


class Actor(nn.Module):
    def __init__(self, s=STATE_DIM, a=ACTION_DIM):
        super().__init__()
        self.net=mlp(s,a)
        nn.init.uniform_(self.net[-1].weight,-3e-3,3e-3)
        nn.init.uniform_(self.net[-1].bias,  -3e-3,3e-3)
    def forward(self,s): return torch.tanh(self.net(s))
    @torch.no_grad()
    def act(self,obs,noise=0.,det=False):
        s=torch.FloatTensor(obs).unsqueeze(0).to(DEVICE)
        a=self.forward(s).squeeze(0).cpu().numpy()
        if not det and noise>0: a+=np.random.normal(0.,noise,a.shape)
        return a.clip(-1.,1.).astype(np.float32)


class Critic(nn.Module):
    def __init__(self, s=STATE_DIM, a=ACTION_DIM):
        super().__init__()
        self.net=mlp(s+a,1)
    def forward(self,s,a): return self.net(torch.cat([s,a],-1))


class ReplayBuffer:
    def __init__(self, s=STATE_DIM, a=ACTION_DIM, cap=300_000):
        self.cap=cap; self.ptr=self.size=0
        self.S =np.zeros((cap,s),np.float32); self.A =np.zeros((cap,a),np.float32)
        self.R =np.zeros((cap,1),np.float32); self.S2=np.zeros((cap,s),np.float32)
        self.D =np.zeros((cap,1),np.float32)
    def add(self,s,a,r,s2,d):
        i=self.ptr%self.cap
        self.S[i]=s;self.A[i]=a;self.R[i]=r;self.S2[i]=s2;self.D[i]=d
        self.ptr=(self.ptr+1)%self.cap; self.size=min(self.size+1,self.cap)
    def sample(self,n):
        idx=np.random.randint(0,self.size,n)
        t=lambda x: torch.FloatTensor(x[idx]).to(DEVICE)
        return t(self.S),t(self.A),t(self.R),t(self.S2),t(self.D)


class DDPGAgent:
    def __init__(self, alr=3e-4, clr=1e-3, gamma=0.99, tau=0.005, batch=256):
        self.gamma=gamma; self.tau=tau; self.batch=batch
        self.actor   =Actor().to(DEVICE); self.actor_t =copy.deepcopy(self.actor)
        self.critic  =Critic().to(DEVICE);self.critic_t=copy.deepcopy(self.critic)
        self.opt_a   =optim.Adam(self.actor.parameters(), lr=alr)
        self.opt_c   =optim.Adam(self.critic.parameters(),lr=clr)
        self.buf     =ReplayBuffer()
        self.c_losses=[]; self.a_losses=[]

    def act(self,obs,noise=0.,det=False): return self.actor.act(obs,noise,det)

    def _soft(self,n,t):
        for p,tp in zip(n.parameters(),t.parameters()):
            tp.data.copy_(self.tau*p.data+(1-self.tau)*tp.data)

    def update(self):
        if self.buf.size<self.batch: return
        S,A,R,S2,D=self.buf.sample(self.batch)
        with torch.no_grad():
            Qt=R+self.gamma*(1-D)*self.critic_t(S2,self.actor_t(S2))
        lc=F.mse_loss(self.critic(S,A),Qt)
        self.opt_c.zero_grad(); lc.backward(); self.opt_c.step()
        la=-self.critic(S,self.actor(S)).mean()
        self.opt_a.zero_grad(); la.backward(); self.opt_a.step()
        self._soft(self.actor,self.actor_t); self._soft(self.critic,self.critic_t)
        self.c_losses.append(float(lc)); self.a_losses.append(float(la))

    def save(self,path):
        torch.save({'actor':self.actor.state_dict(),'critic':self.critic.state_dict()},path)
        print(f'  [SAVED] {path}')

    def load(self,path):
        ck=torch.load(path,map_location=DEVICE)
        self.actor.load_state_dict(ck['actor']); self.actor_t=copy.deepcopy(self.actor)
        self.critic.load_state_dict(ck['critic']); self.critic_t=copy.deepcopy(self.critic)
        print(f'  [LOADED] {path}')

print('DDPG networks defined ✓')

DDPG networks defined ✓


## 8. ▶️ TRAINING CELL
> **Run this cell to train.**  
> Checkpoints are saved to `ckpt_v3/<KEY>.pt`  
> Stats are saved to `ckpt_v3/<KEY>_stats.json` so plotting works independently.

In [11]:
# ═══════════════════════════════════════════════════════════════════
#  TRAINING CONFIGURATION  — edit here
# ═══════════════════════════════════════════════════════════════════
TRAIN_KEYS        = ['H000_030', 'H000_060', 'H000_090', 'H000_180']   # which maneuvers to train
N_EPISODES        = 6000
WARMUP_STEPS      = 3000
NOISE_START       = 0.40
NOISE_END         = 0.04
UPDATES_PER_STEP  = 2
PRINT_EVERY       = 500
EVAL_EVERY        = 1000
N_EVAL            = 30
CURRICULUM_SWITCH = 1000   # episode to switch phase 0→1
SAVE_DIR          = 'ckpt_v3'
# ═══════════════════════════════════════════════════════════════════

os.makedirs(SAVE_DIR, exist_ok=True)
torch.manual_seed(42); np.random.seed(42); random.seed(42)


def _eval_agent(spec, agent, n=30, phase=1):
    env=HeadingChangeEnv(spec,phase); rs,ss,hs,zs,ts,ps=[],[],[],[],[],[]
    for _ in range(n):
        obs=env.reset(); done=False; ep_r=0.
        while not done:
            a=agent.act(obs,det=True); obs,r,done,info=env.step(a); ep_r+=r
        rs.append(ep_r); ss.append(int(info['cause']=='success'))
        hs.append(info['herr']); zs.append(info['z_err'])
        ts.append(info['t']); ps.append(abs(info['phi']))
    return np.mean(rs),np.mean(ss),np.mean(hs),np.mean(zs),np.mean(ts),np.mean(ps)


# ── Training loop ────────────────────────────────────────────────────────────
TRAINED_AGENTS = {}    # key → DDPGAgent  (used later by plotting cell)
ALL_STATS      = {}    # key → stats dict

for SPEC_KEY in TRAIN_KEYS:
    spec  = ALL_SPECS[SPEC_KEY]
    agent = DDPGAgent()
    stats = dict(reward=[],herr=[],z_err=[],success=[],cause=[],
                 alt_viol=[],turn_t=[],phi_peak=[],nz_mean=[])
    best=-np.inf; total=0

    print(f'\n{"═"*65}')
    print(f'  {spec.key}  {spec.label}  [{spec.category.upper()}]')
    print(f'  T_opt={spec.t_opt:.2f}s  T_lim={spec.t_limit:.1f}s')
    print(f'  arc target = ({spec.x_target:.0f}, {spec.y_target:.0f}) m')
    print(f'  Curriculum switch at ep {CURRICULUM_SWITCH}')
    print(f'{"═"*65}')
    t0=time.time()

    for ep in range(1, N_EPISODES+1):
        phase = 0 if ep < CURRICULUM_SWITCH else 1
        env   = HeadingChangeEnv(spec, curriculum_phase=phase)
        obs   = env.reset(); done=False; ep_r=0.
        frac  = min(ep/N_EPISODES, 1.)
        noise = NOISE_START + frac*(NOISE_END-NOISE_START)
        ep_z=[]; ep_phi=[]; ep_nz=[]

        while not done:
            if total < WARMUP_STEPS:
                # Biased warmup: positive turn + minimal altitude trim
                a = np.array([np.random.uniform(0.5,1.0),
                              np.random.uniform(-0.3,0.3),
                              np.random.uniform(-0.2,0.2)], dtype=np.float32)
            else:
                a = agent.act(obs, noise)

            obs2,r,done,info = env.step(a)
            agent.buf.add(obs,a,r,obs2,float(done))
            obs=obs2; ep_r+=r; total+=1
            ep_z.append(info['z_err']); ep_phi.append(abs(info['phi']))
            ep_nz.append(info['nz'])

            if total >= WARMUP_STEPS:
                for _ in range(UPDATES_PER_STEP): agent.update()

        stats['reward'].append(ep_r)
        stats['herr'].append(info['herr'])
        stats['z_err'].append(float(np.mean(ep_z)))
        stats['success'].append(int(info['cause']=='success'))
        stats['cause'].append(info['cause'])
        stats['alt_viol'].append(int(info['cause']=='alt_violation'))
        stats['turn_t'].append(info['t'])
        stats['phi_peak'].append(float(np.max(ep_phi)) if ep_phi else 0.)
        stats['nz_mean'].append(float(np.mean(ep_nz)) if ep_nz else 0.)

        if ep % PRINT_EVERY == 0:
            w=PRINT_EVERY
            ar =np.mean(stats['reward'][-w:])
            sr =100*np.mean(stats['success'][-w:])
            av =100*np.mean(stats['alt_viol'][-w:])
            he =np.mean(stats['herr'][-w:])
            ze =np.mean(stats['z_err'][-w:])
            ph =np.mean(stats['phi_peak'][-w:])
            nzm=np.mean(stats['nz_mean'][-w:])
            tt =np.mean(stats['turn_t'][-w:])
            flag='✓' if sr>0 else '✗'
            print(f'  ep={ep:>5} [ph{phase}] R={ar:>8.1f}  suc={sr:>5.1f}%{flag} '
                  f'altV={av:>5.1f}%  herr={he:>5.1f}°  '
                  f'z={ze:>5.1f}m  φ={ph:>5.1f}°  nz={nzm:>4.1f}g  '
                  f't={tt:>5.1f}s  σ={noise:.3f}  [{time.time()-t0:.0f}s]')

        if ep % EVAL_EVERY == 0:
            er,es,eh,ez,et,ep_=_eval_agent(spec,agent,N_EVAL,phase=1)
            sub5='✓<5s' if et<=5. else f'✗{et:.1f}s'
            print(f'  [EVAL] R={er:.0f} suc={es*100:.0f}% herr={eh:.1f}° '
                  f'z_err={ez:.1f}m t={et:.1f}s φ={ep_:.0f}° {sub5}')
            if er > best:
                best=er
                agent.save(os.path.join(SAVE_DIR, f'{spec.key}.pt'))

    TRAINED_AGENTS[SPEC_KEY] = agent
    ALL_STATS[SPEC_KEY]      = stats

    # Save stats to JSON so plotting cell works independently
    stats_path = os.path.join(SAVE_DIR, f'{SPEC_KEY}_stats.json')
    with open(stats_path,'w') as f:
        json.dump({k: [float(x) for x in v] if isinstance(v[0],(int,float,np.floating,np.integer))
                   else v for k,v in stats.items()}, f)
    print(f'\n[STATS SAVED] {stats_path}')

print('\n✅ Training complete!')


═════════════════════════════════════════════════════════════════
  H000_030  0°→30° (30°)  [NAV]
  T_opt=2.45s  T_lim=15.0s
  arc target = (163, 608) m
  Curriculum switch at ep 1000
═════════════════════════════════════════════════════════════════
  ep=  500 [ph0] R= -8204.6  suc=  0.0%✗ altV= 95.2%  herr= 35.6°  z= 35.9m  φ= 70.5°  nz= 1.5g  t=  6.4s  σ=0.370  [54s]
  ep= 1000 [ph1] R= -7980.0  suc=  0.0%✗ altV= 74.0%  herr= 26.3°  z= 31.9m  φ= 78.3°  nz= 1.8g  t=  7.8s  σ=0.340  [166s]
  [EVAL] R=-7159 suc=0% herr=32.3° z_err=91.7m t=4.5s φ=15° ✓<5s
  [SAVED] ckpt_v3/H000_030.pt
  ep= 1500 [ph1] R= -7131.0  suc=  0.0%✗ altV= 88.2%  herr= 23.1°  z= 33.6m  φ= 78.3°  nz= 1.6g  t=  5.7s  σ=0.310  [250s]
  ep= 2000 [ph1] R= -6091.7  suc=  0.0%✗ altV= 96.4%  herr= 25.7°  z= 34.7m  φ= 75.6°  nz= 1.6g  t=  4.9s  σ=0.280  [321s]
  [EVAL] R=-4370 suc=0% herr=13.9° z_err=56.2m t=4.0s φ=2° ✓<5s
  [SAVED] ckpt_v3/H000_030.pt
  ep= 2500 [ph1] R= -5847.9  suc=  0.0%✗ altV= 98.0%  herr= 21.2°  z=

---
## 9. 📊 PLOTTING CELL
> **This cell is fully independent from the training cell.**  
> It loads `.pt` checkpoint + `_stats.json` from disk.  
> You can restart the kernel and run only this cell to regenerate all plots.

In [12]:
# ═══════════════════════════════════════════════════════════════
#  PLOTTING CONFIG — point to your checkpoint directory
# ═══════════════════════════════════════════════════════════════
PLOT_KEYS = ['H000_030', 'H000_060', 'H000_090', 'H000_180']    # which maneuvers to plot
CKPT_DIR  = 'ckpt_v3'      # where .pt and _stats.json files are
# ═══════════════════════════════════════════════════════════════

plt.rcParams.update({
    'figure.facecolor':'#0d1117','axes.facecolor':'#161b22',
    'axes.edgecolor':'#30363d','axes.labelcolor':'#c9d1d9',
    'xtick.color':'#8b949e','ytick.color':'#8b949e','text.color':'#c9d1d9',
    'grid.color':'#21262d','grid.linewidth':0.6,
    'legend.facecolor':'#161b22','legend.edgecolor':'#30363d',
    'font.family':'monospace',
})
CAT_COL={'nav':'#00d4ff','aggressive':'#fd9644','extreme':'#ff453a'}


def _load_agent(spec_key, ckpt_dir):
    """Load agent from checkpoint. Returns DDPGAgent."""
    agent = DDPGAgent()
    path  = os.path.join(ckpt_dir, f'{spec_key}.pt')
    ck    = torch.load(path, map_location=DEVICE)
    agent.actor.load_state_dict(ck['actor']); agent.actor_t=copy.deepcopy(agent.actor)
    agent.critic.load_state_dict(ck['critic']);agent.critic_t=copy.deepcopy(agent.critic)
    print(f'  [LOADED] {path}')
    return agent


def _load_stats(spec_key, ckpt_dir):
    """Load training stats from JSON. Returns dict."""
    path=os.path.join(ckpt_dir, f'{spec_key}_stats.json')
    with open(path) as f: stats=json.load(f)
    print(f'  [LOADED] {path}')
    return stats


def _record(spec, agent):
    """Roll out one deterministic episode and collect trajectory."""
    env=HeadingChangeEnv(spec,curriculum_phase=1)
    obs=env.reset(); done=False; sv0=env.state
    traj={k:[sv0[k]] for k in sv0}
    traj.update(time=[0.],herr=[abs(herr(sv0['psi'],spec.psi1))],
                z_err=[0.],reward=[],
                rew_alt=[],rew_heading=[],rew_pos=[],
                rew_time=[],rew_bank=[],rew_nz=[])
    while not done:
        a=agent.act(obs,det=True); obs,r,done,info=env.step(a)
        sv=env.state
        for k in sv: traj[k].append(sv[k])
        traj['time'].append(sv['elapsed'])
        traj['herr'].append(info['herr'])
        traj['z_err'].append(info['z_err'])
        traj['reward'].append(r)
        ri=info['rew_info']
        for c in ('alt','heading','pos','time','bank','nz'):
            traj[f'rew_{c}'].append(ri.get(c,0.))
    traj['termination']=info['cause']; traj['spec']=spec
    return traj


def plot_training_curves(stats, spec, save_dir='.'):
    """Plot 1 — training curves: reward, success rate, alt violations, heading error."""
    fig,axes=plt.subplots(2,2,figsize=(14,9))
    fig.patch.set_facecolor('#0d1117')
    col=CAT_COL[spec.category]
    fig.suptitle(f'Training Curves — {spec.label} [{spec.category.upper()}]',
                 fontsize=12,fontweight='bold',color='#e6edf3')

    n=len(stats['reward'])
    xs=np.arange(n)
    w=max(1,n//20)

    def sm(arr): return np.convolve(arr,np.ones(w)/w,'valid')
    def smx(): return np.linspace(0,n,n-w+1)

    # Reward
    ax=axes[0,0]
    ax.plot(xs,stats['reward'],alpha=0.5,color=col,lw=0.5)
    ax.plot(smx(),sm(stats['reward']),color=col,lw=2.)
    ax.set_title('Episode Reward',color='#c9d1d9'); ax.grid(True)
    ax.set_xlabel('Episode'); ax.set_ylabel('Total Reward')

    # Success + AltViol %
    ax=axes[0,1]
    suc=np.array(stats['success'],float)
    av =np.array(stats['alt_viol'],float)
    ax.plot(smx(),sm(suc)*100,color='#32d74b',lw=2.,label='Success %')
    ax.plot(smx(),sm(av)*100, color='#ff453a',lw=2.,ls='--',label='AltViol %')
    ax.set_ylim(0,105); ax.grid(True)
    ax.set_title('Success & Alt Violation Rate',color='#c9d1d9')
    ax.legend(fontsize=8); ax.set_xlabel('Episode'); ax.set_ylabel('%')

    # Heading error
    ax=axes[1,0]
    he=np.array(stats['herr'],float)
    ax.plot(xs,he,alpha=0.15,color='#ffd60a',lw=0.5)
    ax.plot(smx(),sm(he),color='#ffd60a',lw=2.)
    ax.axhline(spec.success_herr,color='#32d74b',lw=1.5,ls='--',
               label=f'success gate {spec.success_herr:.0f}°')
    ax.set_title('Final Heading Error [°]',color='#c9d1d9')
    ax.legend(fontsize=8); ax.grid(True)
    ax.set_xlabel('Episode'); ax.set_ylabel('herr [°]')

    # Bank + nz
    ax=axes[1,1]
    ph=np.array(stats['phi_peak'],float)
    nzm=np.array(stats['nz_mean'],float)
    ax.plot(smx(),sm(ph),color='#bf5af2',lw=2.,label='φ peak [°]')
    ax2=ax.twinx()
    ax2.plot(smx(),sm(nzm),color='#30d158',lw=2.,label='nz mean [g]')
    ax2.axhline(NZ_SWEET_LO,color='#30d158',lw=0.8,ls=':',alpha=0.6)
    ax2.axhline(NZ_MAX,color='#ff453a',lw=1.,ls='--',alpha=0.7)
    ax2.tick_params(colors='#8b949e'); ax2.set_ylabel('nz [g]',color='#8b949e')
    ax.axhline(PHI_AGGR,color='#fd9644',lw=1.,ls='--',alpha=0.7,label=f'{PHI_AGGR:.0f}° aggr')
    ax.set_title('Bank Angle & Load Factor',color='#c9d1d9')
    ax.legend(fontsize=7,loc='upper left'); ax2.legend(fontsize=7,loc='upper right')
    ax.grid(True); ax.set_xlabel('Episode'); ax.set_ylabel('φ [°]')

    plt.tight_layout()
    fname=os.path.join(save_dir,f'training_{spec.key}.png')
    plt.savefig(fname,dpi=130,bbox_inches='tight',facecolor=fig.get_facecolor())
    plt.close()
    display(Image(fname)); print(f'[Saved] {fname}')


def plot_trajectory(traj, save_dir='.'):
    """Plot 2 — full trajectory analysis (8 panels)."""
    spec=traj['spec']; t=np.array(traj['time']); col=CAT_COL[spec.category]
    z_arr=np.array(traj['z']); nz_arr=np.array(traj['nz'])
    sub5=(traj['termination']=='success' and t[-1]<=5.)
    alt_ok=(np.max(np.abs(z_arr-Z_CRUISE))<=Z_BAND)
    nz_ok=(np.max(nz_arr)<=NZ_MAX and np.min(nz_arr)>=NZ_MIN)

    fig=plt.figure(figsize=(24,15)); fig.patch.set_facecolor('#0d1117')
    badges=[traj['termination'].upper()]
    if sub5:   badges.append('✓ SUB-5s')
    if alt_ok: badges.append('✓ ALT≤50m')
    if nz_ok:  badges.append('✓ NZ 3-7g')
    fig.suptitle(
        f"{spec.label} [{spec.category.upper()}]  |  {'  '.join(badges)}  |  "
        f"t={t[-1]:.1f}s  herr={traj['herr'][-1]:.1f}°  "
        f"Δz_max={np.max(np.abs(z_arr-Z_CRUISE)):.1f}m  "
        f"nz_max={np.max(nz_arr):.2f}g",
        fontsize=10,fontweight='bold',color='#e6edf3',y=1.0)

    gs=gridspec.GridSpec(3,4,figure=fig,hspace=0.52,wspace=0.38)
    def ax(r,c,cs=1): return fig.add_subplot(gs[r,c:c+cs])

    # Panel 1: Heading
    a=ax(0,0,2)
    a.plot(t,traj['psi'],color=col,lw=2.2,label='ψ [°]')
    a.axhline(spec.psi1,color='#32d74b',lw=1.5,ls='--',label=f'Target {spec.psi1:.0f}°')
    a.axhline(spec.psi0,color='white',lw=0.8,ls=':',alpha=0.4)
    a.axvline(spec.t_opt,color='#ffd60a',lw=1.,ls=':',label=f'T_opt={spec.t_opt:.1f}s')
    a.axvline(5.,color='#ff453a',lw=1.5,ls='--',label='5s deadline')
    a.set_title('Heading ψ [°]'); a.grid(True); a.legend(fontsize=7); a.set_xlabel('t [s]')

    # Panel 2: Altitude
    a=ax(0,2,2)
    a.plot(t,z_arr,color='#7cfc00',lw=2.5)
    a.axhline(Z_CRUISE,color='white',lw=0.8,ls='--',alpha=0.5)
    a.axhline(Z_CRUISE+Z_BAND,color='#ff453a',lw=2.,ls='--',label='±50m HARD WALL')
    a.axhline(Z_CRUISE-Z_BAND,color='#ff453a',lw=2.,ls='--')
    a.fill_between(t,Z_CRUISE-Z_BAND,Z_CRUISE+Z_BAND,alpha=0.10,color='#32d74b')
    bad=np.abs(z_arr-Z_CRUISE)>Z_BAND
    if bad.any(): a.fill_between(t,z_arr,Z_CRUISE,where=bad,alpha=0.4,color='#ff453a',label='VIOLATION')
    a.set_ylim(Z_CRUISE-200,Z_CRUISE+200)
    a.set_title('Altitude z [m]  ← HARD wall ±50m'); a.grid(True)
    a.legend(fontsize=7); a.set_xlabel('t [s]')

    # Panel 3: Bank
    a=ax(1,0)
    a.plot(t,traj['phi'],color='#bf5af2',lw=2.)
    a.axhspan( PHI_AGGR, PHI_LIMIT,alpha=0.12,color='#fd9644',label=f'≥{PHI_AGGR:.0f}° bonus')
    a.axhspan(-PHI_LIMIT,-PHI_AGGR,alpha=0.12,color='#fd9644')
    a.axhline( PHI_OPT_DEG,color='white',lw=0.8,ls=':',alpha=0.5,label=f'φ_opt={PHI_OPT_DEG:.0f}°')
    a.set_title('Bank φ [°]'); a.grid(True); a.legend(fontsize=7); a.set_xlabel('t [s]')

    # Panel 4: Load factor
    a=ax(1,1)
    a.plot(t,nz_arr,color='#30d158',lw=2.)
    a.axhspan(NZ_SWEET_LO,NZ_SWEET_HI,alpha=0.18,color='#30d158',label=f'{NZ_SWEET_LO}–{NZ_SWEET_HI}g sweet')
    a.axhspan(NZ_MIN,NZ_SWEET_LO,alpha=0.08,color='#ffd60a',label=f'{NZ_MIN}–{NZ_SWEET_LO}g ramp')
    a.axhline(NZ_MAX,color='#ff453a',lw=2.,ls='--',label=f'{NZ_MAX}g LIMIT')
    a.axhline(NZ_MIN,color='#ffd60a',lw=1.2,ls='--',label=f'{NZ_MIN}g floor')
    a.set_ylim(0,9); a.set_title('Load factor nz [g]'); a.grid(True)
    a.legend(fontsize=7); a.set_xlabel('t [s]')

    # Panel 5: Speed
    a=ax(1,2)
    a.plot(t,traj['V'],color='#ff6b35',lw=2.)
    a.axhline(V_CRUISE,color='white',lw=0.8,ls='--',alpha=0.5)
    a.axhline(V_CRUISE-30,color='#ff453a',lw=1.2,ls='--',label='±30m/s band')
    a.axhline(V_CRUISE+30,color='#ff453a',lw=1.2,ls='--')
    a.set_title('TAS V [m/s]'); a.grid(True); a.legend(fontsize=7); a.set_xlabel('t [s]')

    # Panel 6: Reward components
    a=ax(1,3)
    t_a=np.array(traj['time'][1:])
    RMAP={'alt':'#ff375f','heading':'#ffd60a','pos':'#64d2ff',
          'time':'#30d158','bank':'#fd9644','nz':'#bf5af2'}
    for c,cc in RMAP.items():
        arr=traj.get(f'rew_{c}',[])
        if arr: a.plot(t_a,arr,color=cc,lw=1.4,label=c)
    a.axhline(0,color='white',lw=0.4,alpha=0.3)
    a.axvline(5.,color='#ff453a',lw=1.,ls='--',alpha=0.7,label='5s')
    a.set_title('Reward components'); a.grid(True); a.legend(fontsize=6); a.set_xlabel('t [s]')

    # Panel 7: Alt error
    a=ax(2,0)
    ze=np.array(traj['z_err'])
    a.fill_between(t,ze,alpha=0.25,color='#7cfc00')
    a.plot(t,ze,color='#7cfc00',lw=2.)
    a.axhline(Z_BAND,color='#ff453a',lw=2.,ls='--',label='50m WALL')
    a.set_ylim(0,max(Z_BAND*3,ze.max()*1.1 if ze.max()>0 else Z_BAND*3))
    a.set_title('|z − 5000| [m]  must stay < 50m'); a.grid(True)
    a.legend(fontsize=7); a.set_xlabel('t [s]')

    # Panel 8: Heading error
    a=ax(2,1)
    a.fill_between(t,traj['herr'],alpha=0.2,color=col)
    a.plot(t,traj['herr'],color=col,lw=2.)
    a.axhline(spec.success_herr,color='#32d74b',lw=1.5,ls='--',label=f'{spec.success_herr:.0f}° gate')
    a.axvline(5.,color='#ff453a',lw=1.5,ls='--',label='5s')
    a.set_title('Heading error [°]'); a.grid(True); a.legend(fontsize=7); a.set_xlabel('t [s]')

    # Panel 9: Ground track
    a=ax(2,2,2)
    x_km=np.array(traj['x'])/1000; y_km=np.array(traj['y'])/1000
    sc=a.scatter(x_km,y_km,c=t,cmap='plasma',s=15,linewidths=0,zorder=3)
    a.plot(x_km,y_km,color='white',lw=0.5,alpha=0.3)
    a.plot(x_km[0],y_km[0],'o',color='#32d74b',ms=10,label='start',zorder=5)
    a.plot(x_km[-1],y_km[-1],'s',color='#ff453a',ms=10,label='end',zorder=5)
    a.plot(spec.x_target/1000,spec.y_target/1000,'*',color='#ffd60a',ms=16,
           label='arc target ★',zorder=5)
    # Ideal arc overlay
    arc_t=np.linspace(0,spec.delta,60)
    arc_x,arc_y=[],[]
    for ang in arc_t:
        xe,yn=_arc_endpoint(spec.psi0,ang,spec.R)
        arc_x.append(xe/1000); arc_y.append(yn/1000)
    a.plot(arc_x,arc_y,color='#32d74b',lw=1.5,ls=':',alpha=0.7,label='ideal arc')
    plt.colorbar(sc,ax=a,label='t [s]')
    a.set_title('Ground track [km]  dotted=ideal arc'); a.grid(True)
    a.set_aspect('equal','datalim'); a.legend(fontsize=7)
    a.set_xlabel('E [km]'); a.set_ylabel('N [km]')

    fname=os.path.join(save_dir,f'traj_{spec.key}.png')
    plt.savefig(fname,dpi=130,bbox_inches='tight',facecolor=fig.get_facecolor())
    plt.close()
    display(Image(fname)); print(f'[Saved] {fname}')


def constraint_report(traj):
    """Print constraint satisfaction table."""
    spec=traj['spec']; t=np.array(traj['time'])
    z=np.array(traj['z']); nz=np.array(traj['nz'])
    V_arr=np.array(traj['V']); phi=np.array(traj['phi'])
    z_err=np.abs(z-Z_CRUISE)

    psi=np.array(traj['psi']); t_hit=None
    for i,p in enumerate(psi):
        if abs(p-spec.psi1)<spec.success_herr and i>0: t_hit=t[i]; break

    print(f'\n  ┌{"─"*58}┐')
    print(f'  │  CONSTRAINT REPORT: {spec.label:<37}│')
    print(f'  ├{"─"*58}┤')
    #s0=f'{t_hit:.0f}s  {"✓ SUB-5s" if t_hit and t_hit<=5. else "✗ LATE" if t_hit else "NOT REACHED"}'
    #print(f'  │  <5s deadline      : {s0:<36}│')
    s1=f'{z_err.max():.1f}m  {"✓ OK" if z_err.max()<=Z_BAND else "*** VIOLATED ***"}'
    print(f'  │  alt max error     : {s1:<36}│')
    print(f'  │  alt mean error    : {z_err.mean():.1f}m{"":>32}│')
    s2=f'{nz.max():.2f}g  {"✓ OK" if nz.max()<=NZ_MAX else "*** EXCEEDED ***"}'
    print(f'  │  nz max            : {s2:<36}│')
    s3=f'{nz.min():.2f}g  {"✓ OK" if nz.min()>=NZ_MIN else "*** BELOW FLOOR ***"}'
    print(f'  │  nz min            : {s3:<36}│')
    sweet=100*np.mean((nz>=NZ_SWEET_LO)&(nz<=NZ_SWEET_HI))
    print(f'  │  nz sweet-spot %   : {sweet:.1f}% in {NZ_SWEET_LO}–{NZ_SWEET_HI}g{"":>18}│')
    vok=V_arr.min()>=V_CRUISE-30 and V_arr.max()<=V_CRUISE+30
    s4=f'[{V_arr.min():.0f},{V_arr.max():.0f}] m/s  {"✓ OK" if vok else "✗ CHECK"}'
    print(f'  │  speed band        : {s4:<36}│')
    dist_f=math.hypot(traj['x'][-1]-spec.x_target,traj['y'][-1]-spec.y_target)
    s5=f'{dist_f:.0f}m  {"✓ OK" if dist_f<300 else "✗ FAR"}'
    print(f'  │  arc-endpoint dist : {s5:<36}│')
    phi_max=np.max(np.abs(phi))
    s6=f'{phi_max:.1f}°  {"✓ AGGRESSIVE" if phi_max>=PHI_AGGR else "✗ shallow"}'
    print(f'  │  bank max          : {s6:<36}│')
    print(f'  │  termination       : {traj["termination"].upper():<36}│')
    print(f'  └{"─"*58}┘')


# ── Run plotting ─────────────────────────────────────────────────────────────
for key in PLOT_KEYS:
    spec = ALL_SPECS[key]
    print(f'\n📊 Plotting {key} — {spec.label}')

    # Load agent from checkpoint
    agent = _load_agent(key, CKPT_DIR)

    # Load stats from JSON
    stats = _load_stats(key, CKPT_DIR)

    # Plot 1: Training curves
    print('  → Training curves...')
    plot_training_curves(stats, spec, save_dir=CKPT_DIR)

    # Plot 2: Trajectory
    print('  → Trajectory...')
    traj = _record(spec, agent)
    plot_trajectory(traj, save_dir=CKPT_DIR)

    # Constraint report
    constraint_report(traj)

print('\n✅ All plots generated!')

Output hidden; open in https://colab.research.google.com to view.